In [ ]:
%cd /content/paper-theater-ai

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask, save_json
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.scene_builder import parse_florence_boxes, assign_layers
from src.occlusion_heuristic import heuristic_complete
from src.occlusion_amodal import amodal_experimental
from src.mask_cleanup import cleanup_mask

In [ ]:
paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)

In [ ]:
image_path = paths.input_dir / "scene.png"
image = load_image(image_path, max_side=cfg.image_max_side)

plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
florence = FlorenceParser(cfg.florence_model)

caption = florence.get_dense_caption(image)
caption

In [ ]:
det = florence.get_open_vocab_detection(
    image,
    "tree, temple, pagoda, house, building, bridge, mountain, sky, bush, foreground plant"
)
det

In [ ]:
boxes = parse_florence_boxes(det)
boxes[:5]

In [ ]:
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
segmented = segmenter.segment_boxes(image, boxes)

len(segmented), segmented[0].keys()

In [ ]:
sample = segmented[0]
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.imshow(sample["mask"], alpha=0.4)
plt.title(sample["label"])
plt.axis("off")
plt.show()

In [ ]:
depth_runner = DepthRunner(cfg.depth_encoder)
depth = depth_runner.infer(image)

plt.figure(figsize=(8, 8))
plt.imshow(depth, cmap="plasma")
plt.axis("off")
plt.title("Depth map")
plt.show()

In [ ]:
layer_assignments = assign_layers(segmented, depth, cfg.target_num_layers)
layer_assignments

In [ ]:
heuristic_results = []

for i, obj in enumerate(segmented):
    completed = heuristic_complete(obj["mask"], obj["label"])
    cleaned = cleanup_mask(completed, cfg.min_component_area, cfg.smooth_kernel)

    out_path = paths.completed_dir / f"heuristic_{i:03d}.png"
    save_mask(out_path, cleaned)

    heuristic_results.append({
        "index": i,
        "label": obj["label"],
        "bbox": obj["bbox"],
        "layer": layer_assignments[i],
        "mask_path": str(out_path)
    })

heuristic_results[:3]

In [ ]:
from src.occlusion_openai import openai_edit
from PIL import Image

In [ ]:
openai_results = []

for i, obj in enumerate(segmented):
    guess = heuristic_complete(obj["mask"], obj["label"])
    cleaned_mask = cleanup_mask(guess, cfg.min_component_area, cfg.smooth_kernel)

    # use this only on structured labels
    label = obj["label"].lower()
    structured = any(x in label for x in ["temple", "pagoda", "building", "house", "bridge", "roof"])

    if structured:
        edited = openai_edit(image, cleaned_mask, obj["label"], cfg.openai_model)
        out_img = paths.completed_dir / f"openai_edit_{i:03d}.png"
        Image.fromarray(edited).save(out_img)

    out_mask = paths.completed_dir / f"openai_mask_{i:03d}.png"
    save_mask(out_mask, cleaned_mask)

    openai_results.append({
        "index": i,
        "label": obj["label"],
        "bbox": obj["bbox"],
        "layer": layer_assignments[i],
        "mask_path": str(out_mask)
    })

In [ ]:
amodal_results = []

for i, obj in enumerate(segmented):
    completed = amodal_experimental(obj["mask"], obj["label"])
    cleaned = cleanup_mask(completed, cfg.min_component_area, cfg.smooth_kernel)

    out_path = paths.completed_dir / f"amodal_{i:03d}.png"
    save_mask(out_path, cleaned)

    amodal_results.append({
        "index": i,
        "label": obj["label"],
        "bbox": obj["bbox"],
        "layer": layer_assignments[i],
        "mask_path": str(out_path)
    })